<a href="https://colab.research.google.com/github/Light466/My_Programs/blob/main/Sentiment_analysis_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install NLTK
!pip install nltk

In [2]:
import nltk
import pandas as pd

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
data = {
    "text": [
        "I love this movie",
        "This movie is excellent",
        "The product is amazing",
        "I really enjoyed this book",
        "This is a wonderful experience",
        "The service was very good",
        "I am happy with the product",
        "This is fantastic",
        "I hate this movie",
        "This movie is terrible",
        "The product is bad",
        "I really disliked this book",
        "This is a horrible experience",
        "The service was very poor",
        "I am unhappy with the product",
        "This is disappointing"
    ],

    "sentiment": [
        "positive",
        "positive",
        "positive",
        "positive",
        "positive",
        "positive",
        "positive",
        "positive",
        "negative",
        "negative",
        "negative",
        "negative",
        "negative",
        "negative",
        "negative",
        "negative"
    ]
}

df = pd.DataFrame(data)

print(df)

                              text sentiment
0                I love this movie  positive
1          This movie is excellent  positive
2           The product is amazing  positive
3       I really enjoyed this book  positive
4   This is a wonderful experience  positive
5        The service was very good  positive
6      I am happy with the product  positive
7                This is fantastic  positive
8                I hate this movie  negative
9           This movie is terrible  negative
10              The product is bad  negative
11     I really disliked this book  negative
12   This is a horrible experience  negative
13       The service was very poor  negative
14   I am unhappy with the product  negative
15           This is disappointing  negative


In [5]:
# Create stemmer
stemmer = PorterStemmer()

# Get English stop words
stop_words = set(stopwords.words('english'))

def preprocess_text(text):

    # Convert text to lowercase
    text = text.lower()

    # Tokenization
    words = word_tokenize(text)

    # Remove stop words and punctuation
    words = [
        word for word in words
        if word.isalpha() and word not in stop_words
    ]

    # Stemming
    words = [
        stemmer.stem(word)
        for word in words
    ]

    # Join words
    return " ".join(words)


# Apply preprocessing
df["processed_text"] = df["text"].apply(preprocess_text)

print(df[["text", "processed_text", "sentiment"]])

                              text      processed_text sentiment
0                I love this movie           love movi  positive
1          This movie is excellent          movi excel  positive
2           The product is amazing        product amaz  positive
3       I really enjoyed this book   realli enjoy book  positive
4   This is a wonderful experience       wonder experi  positive
5        The service was very good         servic good  positive
6      I am happy with the product       happi product  positive
7                This is fantastic             fantast  positive
8                I hate this movie           hate movi  negative
9           This movie is terrible        movi terribl  negative
10              The product is bad         product bad  negative
11     I really disliked this book  realli dislik book  negative
12   This is a horrible experience      horribl experi  negative
13       The service was very poor         servic poor  negative
14   I am unhappy with th

In [6]:
X = df["processed_text"]
y = df["sentiment"]

print("Input:")
print(X)

print("\nOutput:")
print(y)

Input:
0              love movi
1             movi excel
2           product amaz
3      realli enjoy book
4          wonder experi
5            servic good
6          happi product
7                fantast
8              hate movi
9           movi terribl
10           product bad
11    realli dislik book
12        horribl experi
13           servic poor
14       unhappi product
15            disappoint
Name: processed_text, dtype: object

Output:
0     positive
1     positive
2     positive
3     positive
4     positive
5     positive
6     positive
7     positive
8     negative
9     negative
10    negative
11    negative
12    negative
13    negative
14    negative
15    negative
Name: sentiment, dtype: object


In [7]:
vectorizer = TfidfVectorizer()

X_tfidf = vectorizer.fit_transform(X)

print("TF-IDF shape:", X_tfidf.shape)

TF-IDF shape: (16, 22)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 12
Testing samples: 4


In [9]:
model = MultinomialNB()

model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [10]:
y_pred = model.predict(X_test)

print("Predicted:")
print(y_pred)

print("\nActual:")
print(y_test.values)

Predicted:
['positive' 'negative' 'negative' 'negative']

Actual:
['negative' 'positive' 'negative' 'positive']


In [11]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy percentage:", accuracy * 100, "%")

Accuracy: 0.25
Accuracy percentage: 25.0 %


In [12]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negative       0.33      0.50      0.40         2
    positive       0.00      0.00      0.00         2

    accuracy                           0.25         4
   macro avg       0.17      0.25      0.20         4
weighted avg       0.17      0.25      0.20         4



In [13]:
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["negative", "positive"]
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[1 1]
 [2 0]]


In [15]:
def predict_sentiment(sentence):

    # Preprocess sentence
    processed = preprocess_text(sentence)

    # Convert to TF-IDF
    vector = vectorizer.transform([processed])

    # Predict
    prediction = model.predict(vector)[0]

    return prediction


sentence = input("Enter a sentence: ")

result = predict_sentiment(sentence)

print("Sentiment:", result)

Enter a sentence: The movie is so so
Sentiment: negative
